In [ ]:
# Use a fresh Python >=3.11 environment with the tested runtime.
# %pip install "transtimegrad[examples] @ git+https://github.com/nbukhanchenko/transtimegrad"


In [ ]:
%matplotlib inline

from copy import deepcopy

from diffusers import (
    PNDMScheduler,
    DDIMScheduler,
    DDPMScheduler,
    DPMSolverMultistepScheduler,
    KDPM2DiscreteScheduler,
    DEISMultistepScheduler,
)
from gluonts.dataset.multivariate_grouper import MultivariateGrouper
from gluonts.dataset.repository.datasets import dataset_recipes, get_dataset
from gluonts.dataset.repository.datasets import get_dataset
from gluonts.evaluation.backtest import make_evaluation_predictions
from gluonts.evaluation import MultivariateEvaluator
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

from transtimegrad.model.rnn_timegrad import TimeGradEstimator
from transtimegrad.model.trans_timegrad import TransTimeGradEstimator
from transtimegrad.dataset.repository.datasets import dataset_recipes

In [ ]:
from transtimegrad.dataset.missing import (
    get_max_shape,
    prepare_dataset_with_missing_values,
    prepare_dataset_without_missing_values,
)

def plot(
    target,
    forecast,
    prediction_length,
    prediction_intervals=(50.0, 90.0),
    color="g",
    fname=None,
):
    label_prefix = ""
    rows = 4
    cols = 4
    fig, axs = plt.subplots(rows, cols, figsize=(24, 24))
    axx = axs.ravel()
    seq_len, target_dim = target.shape

    ps = [50.0] + [
        50.0 + f * c / 2.0 for c in prediction_intervals for f in [-1.0, +1.0]
    ]

    percentiles_sorted = sorted(set(ps))

    for dim in range(0, min(rows * cols, target_dim)):
        ax = axx[dim]

        target[-2 * prediction_length :][dim].plot(ax=ax)

        ps_data = [forecast.quantile(p / 100.0)[:, dim] for p in percentiles_sorted]
        i_p50 = len(percentiles_sorted) // 2

        p50_data = ps_data[i_p50]
        p50_series = pd.Series(data=p50_data, index=forecast.index)
        p50_series.plot(color=color, ls="-", label=f"{label_prefix}median", ax=ax)

        for i in range(len(percentiles_sorted) // 2):
            ptile = percentiles_sorted[i]
            alpha = (ptile / 100.0) ** 0.3
            ax.fill_between(
                forecast.index,
                ps_data[i],
                ps_data[-i - 1],
                facecolor=color,
                alpha=alpha,
                interpolate=True,
            )
            # Hack to create labels for the error intervals.
            # Doesn't actually plot anything, because we only pass a single data point
            pd.Series(data=p50_data[:1], index=forecast.index[:1]).plot(
                color=color,
                alpha=alpha,
                linewidth=10,
                label=f"{label_prefix}{100 - ptile * 2}%",
                ax=ax,
            )

    legend = ["observations", "median prediction"] + [
        f"{k}% prediction interval" for k in prediction_intervals
    ][::-1]
    axx[0].legend(legend, loc="upper left")

    if fname is not None:
        plt.savefig(fname, bbox_inches="tight", pad_inches=0.05)

def prepare_dataset(dataset_name):
    dataset = get_dataset(dataset_name, regenerate=False)

    train_grouper = MultivariateGrouper(
        max_target_dim=int(dataset.metadata.feat_static_cat[0].cardinality)
    )
    test_grouper = MultivariateGrouper(
        num_test_dates=int(len(dataset.test) / len(dataset.train)),
        max_target_dim=int(dataset.metadata.feat_static_cat[0].cardinality),
    )

    return {
        "train": train_grouper(dataset.train),
        "test": test_grouper(dataset.test),
        "metadata": dataset.metadata
    }

def prepare_predictor(dataset, mode, max_epochs, accelerator,
                      num_train_timesteps=100, beta_start=1e-4, beta_end=0.1, beta_schedule="linear",
                      context_length_coef=3, num_layers=2, hidden_size=64, lr=1e-3, weight_decay=1e-8, dropout_rate=0.1,
                      lags_seq=[1, 7, 14, 24, 168], num_inference_steps=99, batch_size=64, num_batches_per_epoch=100):
    model = {
        "rnn_timegrad": TimeGradEstimator,
        "trans_timegrad": TransTimeGradEstimator,
    }
    
    scheduler = DEISMultistepScheduler(
        num_train_timesteps=num_train_timesteps,
        beta_start=beta_start,
        beta_end=beta_end,
        beta_schedule=beta_schedule,
    )
    
    estimator = model[mode](
        freq=dataset["metadata"].freq,
        prediction_length=dataset["metadata"].prediction_length,
        input_size=int(dataset["metadata"].feat_static_cat[0].cardinality),
        scheduler=scheduler,
        context_length=context_length_coef*dataset["metadata"].prediction_length,
        num_layers=num_layers,
        hidden_size=hidden_size,
        lr=lr,
        weight_decay=weight_decay,
        dropout_rate=dropout_rate,
        scaling="mean",
        lags_seq=lags_seq,
        num_inference_steps=num_inference_steps,
        batch_size=batch_size,
        num_batches_per_epoch=num_batches_per_epoch,
        trainer_kwargs=dict(max_epochs=max_epochs, accelerator=accelerator, devices="1"),
    )

    return estimator.train(
        training_data=dataset["train"],
        # validation_data=dataset["test"],
        cache_data=True,
        shuffle_buffer_length=1024,
    )

def prepare_metrics(dataset, predictor, num_samples=100):
    evaluator = MultivariateEvaluator(
        quantiles=(np.arange(20) / 20.0)[1:], target_agg_funcs={"sum": np.sum}
    )

    forecast_it, ts_it = make_evaluation_predictions(
        dataset=dataset["test"], predictor=predictor, num_samples=num_samples
    )
    forecasts, targets = list(forecast_it), list(ts_it)
    agg_metric, _ = evaluator(targets, forecasts, num_series=len(dataset["test"]))

    return forecasts, targets, agg_metric

def prepare_statistics(dataset, forecasts, targets, agg_metric, frame=0):
    metrics = {
        "CRPS": "mean_wQuantileLoss",
        "ND": "ND",
        "NRMSE": "NRMSE",
        "MSE": "MSE"
    }

    for name in metrics:
        print(f"{name}: {round(agg_metric[metrics[name]], 4)}")
        print(f"{name + '-Sum'}: {round(agg_metric['m_sum_' + metrics[name]], 4)}")
        print("-" * 64)

    plot(
        target=targets[frame],
        forecast=forecasts[frame],
        prediction_length=dataset["metadata"].prediction_length,
    )
    plt.show()

def prepare_metrics_for_comparison(datasets, missing_value, missing_types, missing_rates):
    metrics = {key: {} for key in datasets.keys()}

    for dataset_name in datasets:
        metrics[dataset_name][missing_value] = {}
        for missing_type in missing_types:
            metrics[dataset_name][missing_value][missing_type] = {
                "CRPS-Sum": [],
                "NRMSE-Sum": []
            }
            for missing_rate in missing_rates:
                metrics[dataset_name][missing_value][missing_type]["CRPS-Sum"].append(
                    round(
                        datasets[dataset_name]["missing_data"][(
                            missing_value, missing_type, missing_rate
                        )]["agg_metric"]["m_sum_mean_wQuantileLoss"], 4
                    )
                )
                metrics[dataset_name][missing_value][missing_type]["NRMSE-Sum"].append(
                    round(
                        datasets[dataset_name]["missing_data"][(
                            missing_value, missing_type, missing_rate
                        )]["agg_metric"]["m_sum_NRMSE"], 4
                    )
                )

    return metrics

In [ ]:
datasets = {
    "solar_nips": {},
    "electricity_nips": {},
    "exchange_rate_nips": {}
}

for dataset_name in datasets:
    dataset = prepare_dataset(dataset_name)
    predictor = prepare_predictor(
        dataset=dataset, mode="rnn_timegrad", max_epochs=40, accelerator="gpu",
        num_train_timesteps=100, beta_start=1e-4, beta_end=0.1, beta_schedule="linear",
        context_length_coef=1, num_layers=3, hidden_size=40, lr=1e-3, weight_decay=1e-8, dropout_rate=0.1,
        lags_seq=[1, 7, 14, 24, 168], num_inference_steps=99, batch_size=64, num_batches_per_epoch=100)
    forecasts, targets, agg_metric = prepare_metrics(dataset, predictor)
    datasets[dataset_name] = {
        "dataset": dataset,
        "predictor": predictor,
        "forecasts": forecasts,
        "targets": targets,
        "agg_metric": agg_metric,
        "missing_data": {}
    }

In [ ]:
prepare_statistics(
    datasets["solar_nips"]["dataset"], datasets["solar_nips"]["forecasts"],
    datasets["solar_nips"]["targets"], datasets["solar_nips"]["agg_metric"]
)

In [ ]:
prepare_statistics(
    datasets["electricity_nips"]["dataset"], datasets["electricity_nips"]["forecasts"],
    datasets["electricity_nips"]["targets"], datasets["electricity_nips"]["agg_metric"]
)

In [ ]:
prepare_statistics(
    datasets["exchange_rate_nips"]["dataset"], datasets["exchange_rate_nips"]["forecasts"],
    datasets["exchange_rate_nips"]["targets"], datasets["exchange_rate_nips"]["agg_metric"]
)

In [ ]:
from pathlib import Path

# Save portable trained predictors with tensor-only weights.
for dataset_name, result in datasets.items():
    result["predictor"].serialize(Path("artifacts/rnn") / dataset_name)


In [ ]:
datasets = {
    "solar_nips": {},
    "electricity_nips": {},
    "exchange_rate_nips": {}
}

for dataset_name in datasets:
    dataset = prepare_dataset(dataset_name)
    predictor = prepare_predictor(
        dataset=dataset, mode="trans_timegrad", max_epochs=200, accelerator="gpu",
        num_train_timesteps=150, beta_start=1e-4, beta_end=0.1, beta_schedule="linear",
        context_length_coef=3, num_layers=2, hidden_size=64, lr=5e-4, weight_decay=1e-8, dropout_rate=0.1,
        lags_seq=[1, 7, 24], num_inference_steps=149, batch_size=32, num_batches_per_epoch=50)
    forecasts, targets, agg_metric = prepare_metrics(dataset, predictor)
    datasets[dataset_name] = {
        "dataset": dataset,
        "predictor": predictor,
        "forecasts": forecasts,
        "targets": targets,
        "agg_metric": agg_metric,
        "missing_data": {}
    }

In [ ]:
prepare_statistics(
    datasets["solar_nips"]["dataset"], datasets["solar_nips"]["forecasts"],
    datasets["solar_nips"]["targets"], datasets["solar_nips"]["agg_metric"]
)

In [ ]:
prepare_statistics(
    datasets["electricity_nips"]["dataset"], datasets["electricity_nips"]["forecasts"],
    datasets["electricity_nips"]["targets"], datasets["electricity_nips"]["agg_metric"]
)

In [ ]:
prepare_statistics(
    datasets["exchange_rate_nips"]["dataset"], datasets["exchange_rate_nips"]["forecasts"],
    datasets["exchange_rate_nips"]["targets"], datasets["exchange_rate_nips"]["agg_metric"]
)

In [ ]:
num_samples = 100
missing_value = np.nan
missing_types = ["random", "time-level", "series-level"]
missing_rates = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

for dataset_name in datasets:
    for missing_type in missing_types:
        for missing_rate in tqdm(missing_rates):
            dataset = datasets[dataset_name]["dataset"]
            predictor = datasets[dataset_name]["predictor"]
            targets = datasets[dataset_name]["targets"]

            dataset_with_missing_values = prepare_dataset_with_missing_values(
                dataset=dataset,
                missing_rate=missing_rate,
                missing_type=missing_type,
                missing_value=missing_value
            )
            dataset_without_missing_values = prepare_dataset_without_missing_values(
                dataset_with_missing_values=dataset_with_missing_values,
                missing_value=missing_value
            )
            evaluator = MultivariateEvaluator(
                quantiles=(np.arange(20) / 20.0)[1:], target_agg_funcs={"sum": np.sum}
            )
            forecast_it, _ = make_evaluation_predictions(
                dataset=dataset_without_missing_values["test"],
                predictor=predictor,
                num_samples=num_samples
            )
            forecasts = list(forecast_it)
            agg_metric, _ = evaluator(
                targets, forecasts, num_series=len(dataset_without_missing_values["test"])
            )


            datasets[dataset_name]["missing_data"][(
                missing_value, missing_type, missing_rate
            )] = {
                "dataset_with_missing_values": dataset_with_missing_values,
                "dataset_without_missing_values": dataset_without_missing_values,
                "forecasts": forecasts,
                "agg_metric": agg_metric
            }

In [ ]:
metrics = prepare_metrics_for_comparison(datasets, missing_value, missing_types, missing_rates)

In [ ]:
titles = {
    "solar_nips": "Solar",
    "electricity_nips": "Electricity",
    "exchange_rate_nips": "Exchange Rate"
}
scores = {
    "CRPS-Sum": "m_sum_mean_wQuantileLoss",
    "NRMSE-Sum": "m_sum_NRMSE"
}
fig, axs = plt.subplots(3, 2, figsize=(12, 16), dpi=512)

for i, dataset_name in enumerate(["solar_nips", "electricity_nips", "exchange_rate_nips"]):
    for j, metric_name in enumerate(["CRPS-Sum", "NRMSE-Sum"]):
        no_missing_forecast_score = [
            round(datasets[dataset_name]["agg_metric"][scores[metric_name]], 4) for _ in missing_rates
        ]
        axs[i, j].plot(missing_rates, no_missing_forecast_score, label="No Missing")
        axs[i, j].plot(missing_rates, metrics[dataset_name][missing_value]["random"][metric_name], label="Random Missing")
        axs[i, j].plot(missing_rates, metrics[dataset_name][missing_value]["time-level"][metric_name], label="Time Level Missing")
        # axs[i, j].plot(missing_rates, metrics[dataset_name][missing_value]["series-level"][metric_name], label="Series Level Missing")
        axs[i, j].set(xlabel="Dropout Rate", ylabel=metric_name)
        axs[i, j].set_title(titles[dataset_name])
        axs[i, j].axis(xmin=0.05, xmax=0.95, ymin=0.)
        axs[i, j].legend()
        axs[i, j].grid()

In [ ]:
dataset_name = "electricity_nips"
missing_type = "random"
missing_rate = 0.2

prepare_statistics(
    datasets[dataset_name]["missing_data"][(missing_value, missing_type, missing_rate)]["dataset_without_missing_values"],
    datasets[dataset_name]["missing_data"][(missing_value, missing_type, missing_rate)]["forecasts"],
    datasets[dataset_name]["targets"],
    datasets[dataset_name]["missing_data"][(missing_value, missing_type, missing_rate)]["agg_metric"]
)

In [ ]:
dataset_name = "electricity_nips"
missing_type = "random"
missing_rate = 0.4

prepare_statistics(
    datasets[dataset_name]["missing_data"][(missing_value, missing_type, missing_rate)]["dataset_without_missing_values"],
    datasets[dataset_name]["missing_data"][(missing_value, missing_type, missing_rate)]["forecasts"],
    datasets[dataset_name]["targets"],
    datasets[dataset_name]["missing_data"][(missing_value, missing_type, missing_rate)]["agg_metric"]
)

In [ ]:
dataset_name = "electricity_nips"
missing_type = "random"
missing_rate = 0.6

prepare_statistics(
    datasets[dataset_name]["missing_data"][(missing_value, missing_type, missing_rate)]["dataset_without_missing_values"],
    datasets[dataset_name]["missing_data"][(missing_value, missing_type, missing_rate)]["forecasts"],
    datasets[dataset_name]["targets"],
    datasets[dataset_name]["missing_data"][(missing_value, missing_type, missing_rate)]["agg_metric"]
)

In [ ]:
dataset_name = "electricity_nips"
missing_type = "random"
missing_rate = 0.8

prepare_statistics(
    datasets[dataset_name]["missing_data"][(missing_value, missing_type, missing_rate)]["dataset_without_missing_values"],
    datasets[dataset_name]["missing_data"][(missing_value, missing_type, missing_rate)]["forecasts"],
    datasets[dataset_name]["targets"],
    datasets[dataset_name]["missing_data"][(missing_value, missing_type, missing_rate)]["agg_metric"]
)

In [ ]:
from pathlib import Path

# Save portable trained predictors with tensor-only weights.
for dataset_name, result in datasets.items():
    result["predictor"].serialize(Path("artifacts/trans") / dataset_name)
